# 🏦 GigScore — Agent 2: XGBoost + Optuna + SHAP
## Behavioural Credit Scoring Engine for India's Informal Economy
**Barclays Hack-O-Hire 2026 · PS4: AI-Powered Alternate Credit Scoring**  
**Team GigScore · Plaksha University · March 28–29, 2026**

> *"Agent 1 built the dataset. Agent 2 builds the brain."*

| Step | Action | Output |
|---|---|---|
| 0 | Imports & setup | — |
| 1 | Load Agent 1 outputs | Clean feature matrix |
| 2 | Handle class imbalance via scale_pos_weight | No resampling |
| 3 | Optuna hyperparameter search (50 trials) | Best XGBoost params |
| 4 | Final model training | Trained GigScore model |
| 5 | Full evaluation suite | AUC, KS, Gini, F1, confusion matrix |
| 6 | SHAP explainability | Per-applicant feature attribution |
| 7 | GigScore 0-100 mapping + tiers | Credit score output |
| 8 | Applicant profiles with SHAP waterfalls | Approve / borderline / decline |
| 9 | Save model + outputs | Handoff to Agent 3 |


## Step 0 — Imports & Setup

In [ ]:
import subprocess
import sys

# Install packages into a clean venv to avoid numba/shap conflicts
pkgs = ['xgboost', 'optuna', 'shap', 'scikit-learn', 'imbalanced-learn',
        'matplotlib', 'seaborn', 'pandas', 'numpy']
for pkg in pkgs:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg,
         '--break-system-packages', '-q'],
        capture_output=True
    )

# Fix numba circular import — must reload before shap
import importlib, types
# Clear any partial numba init
for mod_name in list(sys.modules.keys()):
    if 'numba' in mod_name or 'shap' in mod_name:
        del sys.modules[mod_name]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import pickle
warnings.filterwarnings('ignore')

import xgboost as xgb
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap

from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score, average_precision_score
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy.stats import ks_2samp

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FFFFFF',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})

BARCLAYS_BLUE  = '#00AEEF'
BARCLAYS_NAVY  = '#1F4788'
BARCLAYS_EAGLE = '#D4002A'
GREEN          = '#2ECC71'
ORANGE         = '#F39C12'

print("✅ All packages loaded.")
print(f"   XGBoost {xgb.__version__} | Optuna {optuna.__version__} | SHAP {shap.__version__}")


## Step 1 — Load Agent 1 Outputs

In [ ]:
X_train = pd.read_csv("X_train.csv", index_col=0)
X_test  = pd.read_csv("X_test.csv",  index_col=0)
y_train = pd.read_csv("y_train.csv", index_col=0).squeeze()
y_test  = pd.read_csv("y_test.csv",  index_col=0).squeeze()

print(f"  Train set : {X_train.shape[0]:>6,} rows x {X_train.shape[1]} features")
print(f"  Test  set : {X_test.shape[0]:>6,} rows x {X_test.shape[1]} features")
print(f"  Train default rate : {y_train.mean():.2%}")
print(f"  Test  default rate : {y_test.mean():.2%}")
print(f"  Class imbalance    : {(y_train==0).sum():,} non-default vs {(y_train==1).sum():,} default")
print(f"  Ratio              : {(y_train==0).sum()/(y_train==1).sum():.1f}:1")

upi_cols   = [c for c in X_train.columns if c.startswith('upi_')]
syn_cols   = [c for c in X_train.columns if c.startswith('syn_')]
proxy_cols = [c for c in X_train.columns if c in [
    'income_instability_proxy','credit_seeking_intensity','stress_signal',
    'lifestyle_stability','repayment_stretch','credit_utilization_pressure',
    'digital_engagement_proxy']]
cred_cols  = [c for c in X_train.columns if c.startswith('cred_')]
loan_cols  = [c for c in X_train.columns if c.startswith('loan_')]
core_cols  = [c for c in X_train.columns
              if c not in upi_cols+syn_cols+proxy_cols+cred_cols+loan_cols]

print(f"\n  Feature groups:")
print(f"    Home Credit core  : {len(core_cols):>2}")
print(f"    Credit Score      : {len(cred_cols):>2}")
print(f"    Loan Default      : {len(loan_cols):>2}")
print(f"    UPI behavioral    : {len(upi_cols):>2}")
print(f"    Synthetic UPI     : {len(syn_cols):>2}")
print(f"    Proxy behavioral  : {len(proxy_cols):>2}")
print(f"    TOTAL             : {X_train.shape[1]:>2}")


## Step 2 — Handle Class Imbalance via scale_pos_weight

No SMOTE — we preserve the natural 11% default rate.
XGBoost handles imbalance natively via `scale_pos_weight` which upweights
defaulters during training without changing the data distribution.

**scale_pos_weight = count(non-default) / count(default) ≈ 8.2**


In [ ]:
X_train_sm = X_train.copy()
y_train_sm = y_train.copy()

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"  Class imbalance ratio   : {scale_pos_weight:.2f}:1")
print(f"  scale_pos_weight        : {scale_pos_weight:.2f}")
print(f"  Train samples           : {len(y_train_sm):,} (unchanged — no resampling)")
print(f"  Default rate            : {y_train_sm.mean():.2%} (natural)")
print(f"  XGBoost upweights each defaulter by {scale_pos_weight:.1f}x during training")

fig, ax = plt.subplots(figsize=(7, 4))
counts = y_train.value_counts()
bars = ax.bar(['Non-Default', 'Default'], counts.values,
              color=[BARCLAYS_NAVY, BARCLAYS_EAGLE],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({val/len(y_train):.1%})',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title(f'Natural Class Distribution\nscale_pos_weight = {scale_pos_weight:.1f}x',
             fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('chart_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 3 — Optuna Hyperparameter Search (50 trials)

Bayesian optimisation via TPE — much smarter than grid search.
Learns from previous trials to focus on promising hyperparameter regions.


In [ ]:
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5),
        'scale_pos_weight': scale_pos_weight,
        'random_state':     42,
        'eval_metric':      'auc',
        'verbosity':        0,
    }
    model = XGBClassifier(**params)
    cv    = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    score = cross_val_score(model, X_train_sm, y_train_sm,
                            cv=cv, scoring='roc_auc', n_jobs=-1)
    return score.mean()

print("  Running Optuna 50 trials (~3-5 min)...")
study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
best_cv_auc = study.best_value

print(f"\n  Optuna complete.")
print(f"  Best CV AUC : {best_cv_auc:.4f}")
print(f"  Best params :")
for k, v in best_params.items():
    print(f"    {k:25s}: {v}")

# Optuna charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Optuna Hyperparameter Search", fontsize=13, fontweight='bold')

ax = axes[0]
trial_aucs = [t.value for t in study.trials]
ax.plot(trial_aucs, color=BARCLAYS_NAVY, linewidth=1.5, alpha=0.7, label='Trial AUC')
ax.plot(np.maximum.accumulate(trial_aucs), color=BARCLAYS_EAGLE,
        linewidth=2.5, label='Best so far')
ax.set_xlabel('Trial')
ax.set_ylabel('CV AUC')
ax.set_title('Optimisation History', fontweight='bold')
ax.legend()

ax = axes[1]
param_imp = optuna.importance.get_param_importances(study)
ax.barh(list(param_imp.keys())[::-1], list(param_imp.values())[::-1],
        color=BARCLAYS_NAVY, edgecolor='white')
ax.set_xlabel('Importance')
ax.set_title('Hyperparameter Importance', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_optuna.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Optuna chart saved.")


## Step 4 — Train Final GigScore Model

In [ ]:
final_params = {**best_params,
                'scale_pos_weight': scale_pos_weight,
                'random_state':     42,
                'eval_metric':      'auc',
                'verbosity':        0}
final_params.pop('use_label_encoder', None)

gigscore_model = XGBClassifier(**final_params)
gigscore_model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_prob_train = gigscore_model.predict_proba(X_train)[:, 1]
y_prob_test  = gigscore_model.predict_proba(X_test)[:, 1]
y_pred_test  = (y_prob_test > 0.5).astype(int)

auc_train = roc_auc_score(y_train, y_prob_train)
auc_test  = roc_auc_score(y_test,  y_prob_test)

print(f"  GigScore model trained.")
print(f"  Train AUC  : {auc_train:.4f}")
print(f"  Test  AUC  : {auc_test:.4f}")
print(f"  Overfit gap: {auc_train - auc_test:.4f} {'✅ OK' if auc_train - auc_test < 0.05 else '⚠ CHECK'}")
print(f"\n  Classification Report (threshold = 0.5):")
print(classification_report(y_test, y_pred_test,
                             target_names=['Non-Default', 'Default']))


## Step 5 — Full Evaluation Suite

In [ ]:
scores_default    = y_prob_test[y_test == 1]
scores_nondefault = y_prob_test[y_test == 0]
ks_stat, _        = ks_2samp(scores_nondefault, scores_default)
gini              = 2 * auc_test - 1
avg_prec          = average_precision_score(y_test, y_prob_test)

prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, y_prob_test)
f1_arr     = 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-9)
opt_thresh = thresh_arr[np.argmax(f1_arr[:-1])]
y_pred_opt = (y_prob_test > opt_thresh).astype(int)
f1_opt     = f1_score(y_test,  y_pred_opt)
prec_opt   = precision_score(y_test, y_pred_opt)
rec_opt    = recall_score(y_test,  y_pred_opt)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║          GigScore Model — Full Evaluation Suite                  ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  AUC (ROC)              : {auc_test:.4f}                          ║")
print(f"║  Gini Coefficient       : {gini:.4f}                          ║")
print(f"║  KS Statistic           : {ks_stat:.4f}  {'✅ GOOD' if ks_stat > 0.30 else '⚠ MODERATE':15s}              ║")
print(f"║  Average Precision (AP) : {avg_prec:.4f}                          ║")
print(f"╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Optimal threshold      : {opt_thresh:.4f}                          ║")
print(f"║  Precision @ opt thresh : {prec_opt:.4f}                          ║")
print(f"║  Recall    @ opt thresh : {rec_opt:.4f}                          ║")
print(f"║  F1        @ opt thresh : {f1_opt:.4f}                          ║")
print("╚══════════════════════════════════════════════════════════════════╝")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("GigScore Model — Full Evaluation Suite", fontsize=15, fontweight='bold')

# ROC
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
ax.plot(fpr, tpr, color=BARCLAYS_EAGLE, linewidth=2.5,
        label=f'GigScore (AUC = {auc_test:.4f})')
ax.fill_between(fpr, tpr, fpr, alpha=0.1, color=BARCLAYS_EAGLE)
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(fontsize=10)
ax.text(0.55, 0.15, f'Gini = {gini:.4f}\nKS   = {ks_stat:.4f}',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# PR Curve
ax = axes[0, 1]
ax.plot(rec_arr, prec_arr, color=BARCLAYS_NAVY, linewidth=2.5,
        label=f'GigScore (AP = {avg_prec:.4f})')
ax.axhline(y_test.mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Baseline ({y_test.mean():.2%})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontweight='bold')
ax.legend(fontsize=9)

# Confusion Matrix
ax = axes[0, 2]
cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=ax,
            xticklabels=['Non-Default','Default'],
            yticklabels=['Non-Default','Default'], cbar=False)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix (thresh={opt_thresh:.2f})', fontweight='bold')

# Score Distribution
ax = axes[1, 0]
ax.hist(scores_nondefault, bins=50, alpha=0.7, color=BARCLAYS_NAVY,
        label='Non-Default', density=True)
ax.hist(scores_default,    bins=50, alpha=0.7, color=BARCLAYS_EAGLE,
        label='Default',     density=True)
ax.axvline(opt_thresh, color='black', linestyle='--', linewidth=2,
           label=f'Threshold ({opt_thresh:.2f})')
ax.set_xlabel('Predicted Default Probability')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by Class', fontweight='bold')
ax.legend(fontsize=9)

# KS Chart
ax = axes[1, 1]
sorted_scores = np.sort(y_prob_test)
cum_def       = np.array([np.mean(scores_default    <= s) for s in sorted_scores])
cum_nondef    = np.array([np.mean(scores_nondefault <= s) for s in sorted_scores])
ax.plot(sorted_scores, cum_def,    color=BARCLAYS_EAGLE, linewidth=2, label='Default CDF')
ax.plot(sorted_scores, cum_nondef, color=BARCLAYS_NAVY,  linewidth=2, label='Non-Default CDF')
ks_idx = np.argmax(np.abs(cum_def - cum_nondef))
ax.vlines(sorted_scores[ks_idx], cum_nondef[ks_idx], cum_def[ks_idx],
          color='black', linewidth=2.5, linestyle='--', label=f'KS={ks_stat:.4f}')
ax.set_xlabel('Score')
ax.set_ylabel('Cumulative %')
ax.set_title('KS Chart', fontweight='bold')
ax.legend(fontsize=9)

# Calibration
ax = axes[1, 2]
prob_true, prob_pred = calibration_curve(y_test, y_prob_test, n_bins=10)
ax.plot(prob_pred, prob_true, 's-', color=BARCLAYS_EAGLE, linewidth=2, label='GigScore')
ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Perfect')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('chart_evaluation_suite.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Evaluation suite saved.")


## Step 6 — SHAP Explainability

SHAP assigns each feature a contribution value for each prediction.
This is what makes GigScore a **product** — lenders can explain every decision.

> *"Why was Ramesh declined? His EXT_SOURCE score is 0.28, credit-income ratio is 4.2x,
> and spending velocity spiked 3 months ago — likely an income shock."*


In [ ]:
# TreeExplainer — fast and exact for XGBoost
explainer   = shap.TreeExplainer(gigscore_model)
shap_values = explainer.shap_values(X_test)

shap_importance = pd.DataFrame({
    'feature':   X_test.columns,
    'mean_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_shap', ascending=False).reset_index(drop=True)

print(f"  SHAP values computed: {shap_values.shape}")
print(f"\n  Top 15 features by mean |SHAP|:")
print(shap_importance.head(15).to_string(index=False))


In [ ]:
def feature_colour(fname):
    if fname.startswith('upi_'):  return BARCLAYS_BLUE
    if fname.startswith('syn_'):  return GREEN
    if fname.startswith('cred_'): return ORANGE
    if fname.startswith('loan_'): return '#9B59B6'
    if fname in ['income_instability_proxy','credit_seeking_intensity',
                 'stress_signal','lifestyle_stability','repayment_stretch',
                 'credit_utilization_pressure','digital_engagement_proxy']:
        return GREEN
    return BARCLAYS_NAVY

legend_patches = [
    mpatches.Patch(color=BARCLAYS_NAVY, label='Home Credit core'),
    mpatches.Patch(color=BARCLAYS_BLUE, label='UPI behavioral'),
    mpatches.Patch(color=GREEN,         label='Synthetic/Proxy behavioral'),
    mpatches.Patch(color=ORANGE,        label='Credit Score features'),
    mpatches.Patch(color='#9B59B6',     label='Loan Default features'),
]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("GigScore SHAP Explainability\nWhat drives each credit decision?",
             fontsize=14, fontweight='bold')

# Global importance
ax = axes[0]
top15   = shap_importance.head(15)
colours = [feature_colour(f) for f in top15['feature']]
ax.barh(top15['feature'][::-1], top15['mean_shap'][::-1],
        color=colours[::-1], edgecolor='white')
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title('Global Feature Importance\nMean absolute SHAP', fontweight='bold')
ax.legend(handles=legend_patches, fontsize=9, loc='lower right')

# Beeswarm
ax = axes[1]
top10_feats = shap_importance.head(10)['feature'].tolist()
np.random.seed(42)
for i, feat in enumerate(top10_feats[::-1]):
    feat_idx = X_test.columns.get_loc(feat)
    sv       = shap_values[:, feat_idx]
    fv       = X_test[feat].values
    fv_norm  = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)
    jitter   = np.random.normal(0, 0.08, len(sv))
    sc = ax.scatter(sv, np.full_like(sv, i) + jitter,
                    c=fv_norm, cmap='RdYlBu_r',
                    alpha=0.4, s=8, vmin=0, vmax=1)

ax.set_yticks(range(len(top10_feats)))
ax.set_yticklabels(top10_feats[::-1], fontsize=9)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('SHAP Value', fontsize=11)
ax.set_title('SHAP Beeswarm\nBlue=low value, Red=high value', fontweight='bold')
plt.colorbar(sc, ax=ax, label='Feature value (normalised)')

plt.tight_layout()
plt.savefig('chart_shap_global.png', dpi=150, bbox_inches='tight')
plt.show()
print("  SHAP global chart saved.")


## Step 7 — GigScore 0-100 Mapping & Tier Assignment

In [ ]:
def prob_to_gigscore(prob, k=8, midpoint=0.15):
    score = 100 / (1 + np.exp(k * (prob - midpoint)))
    return np.clip(score, 0, 100)

def assign_tier(score):
    if score >= 80:   return 'Prime'
    elif score >= 65: return 'Near-Prime'
    elif score >= 50: return 'Sub-Prime'
    else:             return 'Decline'

gigscore_test  = prob_to_gigscore(y_prob_test)
gigscore_train = prob_to_gigscore(y_prob_train)
tiers_test     = pd.Series([assign_tier(s) for s in gigscore_test],
                            index=X_test.index, name='tier')

tier_order   = ['Prime', 'Near-Prime', 'Sub-Prime', 'Decline']
tier_counts  = tiers_test.value_counts().reindex(tier_order).dropna()
tier_default = pd.Series({
    t: y_test[tiers_test == t].mean()
    for t in tier_order if (tiers_test == t).sum() > 0
})

print("  GigScore Tier Distribution:")
print(f"  {'Tier':<12} {'Count':>7} {'%':>6} {'Default Rate':>13}")
print("  " + "-"*42)
for t in tier_order:
    if t in tier_counts.index:
        n   = int(tier_counts[t])
        pct = n / len(tiers_test)
        dr  = tier_default.get(t, 0)
        print(f"  {t:<12} {n:>7,} {pct:>6.1%} {dr:>13.2%}")

print(f"\n  mean={gigscore_test.mean():.1f} | median={np.median(gigscore_test):.1f} | std={gigscore_test.std():.1f}")


In [ ]:
tier_colours = {'Prime':'#166534','Near-Prime':BARCLAYS_NAVY,
                'Sub-Prime':BARCLAYS_EAGLE,'Decline':'#9CA3AF'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("GigScore Distribution & Tier Analysis", fontsize=14, fontweight='bold')

# Distribution
ax = axes[0]
n, bins, patches_hist = ax.hist(gigscore_test, bins=50, edgecolor='white', linewidth=0.5)
for patch, left in zip(patches_hist, bins):
    if   left >= 80: patch.set_facecolor('#166534')
    elif left >= 65: patch.set_facecolor(BARCLAYS_NAVY)
    elif left >= 50: patch.set_facecolor(BARCLAYS_EAGLE)
    else:            patch.set_facecolor('#9CA3AF')
for thresh, label, col in [(80,'Prime','#166534'),(65,'Near-Prime',BARCLAYS_NAVY),(50,'Sub-Prime',BARCLAYS_EAGLE)]:
    ax.axvline(thresh, color=col, linestyle='--', linewidth=2, label=label)
total = len(gigscore_test)
text  = '\n'.join([f"{t}: {int(tier_counts[t]):,} ({int(tier_counts[t])/total:.0%})"
                   for t in tier_order if t in tier_counts.index])
ax.text(0.02, 0.95, text, transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('GigScore (0-100)')
ax.set_ylabel('Number of Applicants')
ax.set_title('GigScore Distribution', fontweight='bold')
ax.legend(fontsize=9)

# Default rate by tier
ax = axes[1]
tiers_present = [t for t in tier_order if t in tier_default.index]
bars = ax.bar(tiers_present,
              [tier_default[t]*100 for t in tiers_present],
              color=[tier_colours[t] for t in tiers_present], edgecolor='white')
for bar, t in zip(bars, tiers_present):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{tier_default[t]:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.axhline(y_test.mean()*100, color='gray', linestyle='--', linewidth=1.5,
           label=f'Overall avg ({y_test.mean():.1%})')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by Tier\nMonotonic = well-calibrated', fontweight='bold')
ax.legend(fontsize=9)

# Scatter
ax = axes[2]
sample_idx = np.random.choice(len(gigscore_test), min(2000, len(gigscore_test)), replace=False)
colours_s  = [BARCLAYS_EAGLE if y == 1 else BARCLAYS_NAVY for y in y_test.iloc[sample_idx]]
ax.scatter(y_prob_test[sample_idx], gigscore_test[sample_idx],
           c=colours_s, alpha=0.3, s=8)
ax.set_xlabel('Raw Default Probability')
ax.set_ylabel('GigScore (0-100)')
ax.set_title('Probability → GigScore Mapping', fontweight='bold')
ax.legend(handles=[mpatches.Patch(color=BARCLAYS_EAGLE, label='Default'),
                   mpatches.Patch(color=BARCLAYS_NAVY,  label='Non-Default')], fontsize=9)

plt.tight_layout()
plt.savefig('chart_gigscore_tiers.png', dpi=150, bbox_inches='tight')
plt.show()
print("  GigScore tier chart saved.")


## Step 8 — Applicant Profiles with SHAP Waterfall

Three real applicants with individual SHAP explanations.
This is the **demo moment** — shows exactly why each person got their score.


In [ ]:
gigscore_series = pd.Series(gigscore_test, index=X_test.index)

approved_mask = (gigscore_series >= gigscore_series.quantile(0.90)) & (y_test == 0)
approved_idx  = gigscore_series[approved_mask].idxmax()

border_mask   = (gigscore_series >= 45) & (gigscore_series <= 55)
border_idx    = (gigscore_series[border_mask].sample(1, random_state=42).index[0]
                 if border_mask.sum() > 0
                 else gigscore_series.iloc[(gigscore_series-50).abs().argsort()[:1]].index[0])

declined_mask = (gigscore_series <= gigscore_series.quantile(0.10)) & (y_test == 1)
declined_idx  = (gigscore_series[declined_mask].idxmin()
                 if declined_mask.sum() > 0
                 else gigscore_series.idxmin())

profiles = {
    '✅ APPROVED':   approved_idx,
    '⚠️ BORDERLINE': border_idx,
    '❌ DECLINED':   declined_idx,
}

print("  Three applicant profiles:")
for label, idx in profiles.items():
    gs   = gigscore_series[idx]
    prob = y_prob_test[X_test.index.get_loc(idx)]
    act  = y_test[idx]
    tier = assign_tier(gs)
    print(f"  {label:20s} GigScore: {gs:5.1f} | Tier: {tier:<12} | "
          f"Prob: {prob:.3f} | Actual: {'Default' if act==1 else 'Non-Default'}")


In [ ]:
# SHAP waterfall for each profile
fig, axes = plt.subplots(1, 3, figsize=(21, 9))
fig.suptitle("GigScore Applicant Profiles — SHAP Explanation\nWhy did each applicant get their score?",
             fontsize=14, fontweight='bold')

profile_colours = {
    '✅ APPROVED':   '#166534',
    '⚠️ BORDERLINE': ORANGE,
    '❌ DECLINED':   BARCLAYS_EAGLE,
}

for ax, (label, idx) in zip(axes, profiles.items()):
    test_pos = X_test.index.get_loc(idx)
    sv       = shap_values[test_pos]
    gs       = gigscore_series[idx]
    prob     = y_prob_test[test_pos]
    tier     = assign_tier(gs)
    act      = y_test[idx]

    # Top 10 features by |SHAP| for this specific applicant
    shap_df  = pd.DataFrame({
        'feature': X_test.columns,
        'shap':    sv,
        'value':   X_test.iloc[test_pos].values
    }).reindex(pd.Series(np.abs(sv)).sort_values(ascending=False).index).head(10)
    top10    = shap_df.iloc[::-1]  # reverse for barh

    colours_bar = [BARCLAYS_EAGLE if v > 0 else BARCLAYS_NAVY for v in top10['shap']]
    ax.barh(range(len(top10)), top10['shap'],
            color=colours_bar, edgecolor='white', linewidth=0.5)

    feat_labels = [f"{f}\n= {v:.3f}" for f, v in zip(top10['feature'], top10['value'])]
    ax.set_yticks(range(len(top10)))
    ax.set_yticklabels(feat_labels, fontsize=8)
    ax.axvline(0, color='black', linewidth=1)

    colour = profile_colours[label]
    ax.set_title(
        f"{label}\nGigScore: {gs:.0f}/100 | Tier: {tier}\n"
        f"Prob: {prob:.3f} | Actual: {'Default' if act==1 else 'Non-Default'}",
        fontweight='bold', color=colour, fontsize=11)
    ax.set_xlabel('SHAP Value (red=increases risk, blue=decreases risk)')

    red_p  = mpatches.Patch(color=BARCLAYS_EAGLE, label='Increases default risk')
    blue_p = mpatches.Patch(color=BARCLAYS_NAVY,  label='Decreases default risk')
    ax.legend(handles=[red_p, blue_p], fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('chart_shap_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print("  SHAP profile chart saved.")


## Step 9 — Save Model & Outputs for Agent 3

In [ ]:
# Save model
with open('gigscore_model.pkl', 'wb') as f:
    pickle.dump(gigscore_model, f)
gigscore_model.save_model('gigscore_model.json')

# Predictions
scores_df = pd.DataFrame({
    'applicant_id':      X_test.index,
    'gigscore':          gigscore_test,
    'default_prob':      y_prob_test,
    'tier':              tiers_test.values,
    'actual_default':    y_test.values,
    'predicted_default': y_pred_opt,
})
scores_df.to_csv('GigScore_Predictions.csv', index=False)

# SHAP importance
shap_importance.to_csv('GigScore_SHAP_Importance.csv', index=False)

# Top 3 SHAP per applicant for Agent 3 explanation cards
top3_records = []
for i in range(len(X_test)):
    sv      = shap_values[i]
    top_idx = np.abs(sv).argsort()[::-1][:3]
    top3_records.append({
        'applicant_id': X_test.index[i],
        'gigscore':     gigscore_test[i],
        'tier':         tiers_test.iloc[i],
        'feature_1':    X_test.columns[top_idx[0]], 'shap_1': sv[top_idx[0]],
        'feature_2':    X_test.columns[top_idx[1]], 'shap_2': sv[top_idx[1]],
        'feature_3':    X_test.columns[top_idx[2]], 'shap_3': sv[top_idx[2]],
    })
pd.DataFrame(top3_records).to_csv('GigScore_SHAP_Top3.csv', index=False)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║              AGENT 2 COMPLETE — SUMMARY                         ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Model              : XGBoost (Optuna-tuned)                    ║")
print(f"║  Class handling     : scale_pos_weight = {scale_pos_weight:.2f} (no SMOTE)       ║")
print(f"║  Test AUC           : {auc_test:.4f}                               ║")
print(f"║  Gini Coefficient   : {gini:.4f}                               ║")
print(f"║  KS Statistic       : {ks_stat:.4f} {'✅' if ks_stat > 0.30 else '⚠'}                              ║")
print(f"║  Optimal Threshold  : {opt_thresh:.4f}                               ║")
print(f"║  F1 @ opt threshold : {f1_opt:.4f}                               ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  GigScore Tier Distribution                                      ║")
for t in tier_order:
    if t in tier_counts.index:
        n_t = int(tier_counts[t])
        dr  = tier_default.get(t, 0)
        print(f"║    {t:<12}: {n_t:>5,} ({n_t/len(tiers_test):.0%}) | default rate: {dr:.2%}         ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  FILES SAVED                                                     ║")
print("║  📦 gigscore_model.pkl            → trained model               ║")
print("║  📦 gigscore_model.json           → XGBoost native format       ║")
print("║  📄 GigScore_Predictions.csv      → scores + tiers              ║")
print("║  📄 GigScore_SHAP_Importance.csv  → global feature importance   ║")
print("║  📄 GigScore_SHAP_Top3.csv        → top 3 SHAP per applicant    ║")
print("║  📊 7 charts                                                     ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  NEXT: Agent 3 — GigScore Card UI + Lender Dashboard            ║")
print("╚══════════════════════════════════════════════════════════════════╝")
